# INV-M365-CE — Authoritative Persona Post-H5 Certification Parity Scope Correction v1

**Purpose:** Prove the unified post-H5 certification boundary for persona certification, department certification, and enterprise release-gate truth.

**Lemma mapping:** `docs/ma/lemmas/L83_m365_authoritative_persona_post_h5_certification_parity_scope_correction_v1.md`

**Invariant support:** `invariants/lemmas/L83_m365_authoritative_persona_post_h5_certification_parity_scope_correction_v1.yaml`

**Expected results:** Deterministically compute the final `59 total / 54 active / 5 planned / 430 actions` certification targets while preserving the existing department-pack status taxonomy because department-pack authority files remain out of scope in this phase.


This notebook is the Phase 5 source for the unified certification correction. It does not modify runtime roster truth. It derives the only certification state that can reconcile to the already-truthful final registry, activated surface, and packaging surface.


In [ ]:
from __future__ import annotations

import hashlib
import json
from collections import Counter
from pathlib import Path

import yaml

repo = Path.cwd().resolve()
while not (repo / "registry" / "persona_registry_v2.yaml").exists():
    if repo.parent == repo:
        raise RuntimeError("repo root not found")
    repo = repo.parent

paths = {
    "registry": repo / "registry" / "persona_registry_v2.yaml",
    "persona_certification": repo / "registry" / "persona_certification_v1.yaml",
    "department_certification": repo / "registry" / "department_certification_v1.yaml",
    "release_gate": repo / "registry" / "enterprise_release_gate_v2.yaml",
    "workload_certification": repo / "registry" / "workload_certification_v1.yaml",
    "activated_surface": repo / "registry" / "activated_persona_surface_v1.yaml",
    "workforce_packaging": repo / "registry" / "workforce_packaging_v1.yaml",
    "scope_gap_verification": repo / "configs" / "generated" / "authoritative_persona_post_h5_parity_correction_v1_verification.json",
    "verification_output": repo / "configs" / "generated" / "authoritative_persona_post_h5_certification_parity_scope_correction_v1_verification.json",
}

required_input_keys = {
    "registry",
    "persona_certification",
    "department_certification",
    "release_gate",
    "workload_certification",
    "activated_surface",
    "workforce_packaging",
    "scope_gap_verification",
}
for key in required_input_keys:
    assert paths[key].exists(), f"missing {key}: {paths[key]}"

department_pack_paths = sorted((repo / "registry").glob("department_pack_*_v1.yaml"))
assert department_pack_paths, "expected department pack authority files"


Load the live authoritative surfaces. The registry, activated surface, and packaging surface are already final; the certification surfaces remain stale and are what this notebook must reconcile.


In [ ]:
registry = yaml.safe_load(paths["registry"].read_text(encoding="utf-8"))
persona_certification = yaml.safe_load(paths["persona_certification"].read_text(encoding="utf-8"))
department_certification = yaml.safe_load(paths["department_certification"].read_text(encoding="utf-8"))
release_gate = yaml.safe_load(paths["release_gate"].read_text(encoding="utf-8"))
workload_certification = yaml.safe_load(paths["workload_certification"].read_text(encoding="utf-8"))
activated_surface = yaml.safe_load(paths["activated_surface"].read_text(encoding="utf-8"))
workforce_packaging = yaml.safe_load(paths["workforce_packaging"].read_text(encoding="utf-8"))
scope_gap_verification = json.loads(paths["scope_gap_verification"].read_text(encoding="utf-8"))
department_packs = {
    yaml.safe_load(path.read_text(encoding="utf-8"))["department"]["id"]: yaml.safe_load(path.read_text(encoding="utf-8"))
    for path in department_pack_paths
}

registry_summary = dict(registry["summary"])
risk_distribution = dict(sorted(Counter(persona["risk_tier"] for persona in registry["personas"].values()).items()))

department_counts = {}
for persona_id, persona in registry["personas"].items():
    department_id = persona["department"]
    stats = department_counts.setdefault(
        department_id,
        {
            "persona_count": 0,
            "active_persona_count": 0,
            "planned_persona_count": 0,
            "registry_backed_persona_count": 0,
            "contract_only_persona_count": 0,
        },
    )
    stats["persona_count"] += 1
    if persona["status"] == "active":
        stats["active_persona_count"] += 1
    else:
        stats["planned_persona_count"] += 1
    if persona["coverage_status"] == "registry-backed":
        stats["registry_backed_persona_count"] += 1
    else:
        stats["contract_only_persona_count"] += 1

department_status_targets = {}
for department_id, pack in sorted(department_packs.items()):
    department_status_targets[department_id] = {
        **department_counts[department_id],
        "workflow_family_count": len(pack.get("workflow_families", [])),
        "workload_family_count": len(pack.get("workload_families", [])),
        "department_status": pack["department"]["status"],
        "certification_verdict": "certified",
    }

workload_kpis = workload_certification["kpis"]
activated_kpis = activated_surface["kpis"]
packaging_kpis = workforce_packaging["kpis"]

final_persona_certification_targets = {
    "total_personas": registry_summary["total_personas"],
    "active_personas": registry_summary["active_personas"],
    "planned_personas": registry_summary["planned_personas"],
    "registry_backed_personas": registry_summary["registry_backed_personas"],
    "contract_only_personas": registry_summary["persona_contract_only_personas"],
    "risk_tier_critical": risk_distribution["critical"],
    "risk_tier_high": risk_distribution["high"],
    "risk_tier_medium": risk_distribution["medium"],
    "risk_tier_low": risk_distribution["low"],
}

final_department_certification_targets = {
    "total_departments": len(department_status_targets),
    "total_department_personas": registry_summary["total_personas"],
    "active_department_personas": registry_summary["active_personas"],
    "planned_department_personas": registry_summary["planned_personas"],
    "registry_backed_department_personas": registry_summary["registry_backed_personas"],
    "contract_only_department_personas": registry_summary["persona_contract_only_personas"],
    "department_statuses": department_status_targets,
}

final_release_gate_targets = {
    "workload_domains_certified": workload_kpis["domains_with_routed_actions"],
    "workload_domains_not_yet": workload_kpis["domains_without_routed_actions"],
    "personas_certified": registry_summary["total_personas"],
    "active_personas_certified": registry_summary["active_personas"],
    "planned_personas_certified": registry_summary["planned_personas"],
    "departments_certified": len(department_status_targets),
    "total_routed_actions": packaging_kpis["total_routed_actions"],
}

residual_gaps = [
    "2 executor domains (workmanagement, publishing) have zero routed actions",
    "5 of 59 personas remain contract-only deferred external-platform marketing roles with zero actions",
    "Department pack status fields remain the bounded H4S taxonomy and are not reclassified in this phase",
    "Cross-department workflows are contract-certified, not live-executed",
]

verification = {
    "status": "passed",
    "lemma_id": "L83",
    "predecessor_scope_gap_proof": scope_gap_verification["scope_correction_required"],
    "registry_summary": registry_summary,
    "risk_distribution": final_persona_certification_targets | {
        "risk_tier_critical": final_persona_certification_targets["risk_tier_critical"],
        "risk_tier_high": final_persona_certification_targets["risk_tier_high"],
        "risk_tier_medium": final_persona_certification_targets["risk_tier_medium"],
        "risk_tier_low": final_persona_certification_targets["risk_tier_low"],
    },
    "current_stale_surface_counts": {
        "persona_certification": dict(persona_certification["kpis"]),
        "department_certification": dict(department_certification["kpis"]),
        "enterprise_release_gate": dict(release_gate["kpis"]),
    },
    "final_persona_certification_targets": final_persona_certification_targets,
    "final_department_certification_targets": final_department_certification_targets,
    "final_release_gate_targets": final_release_gate_targets,
    "activated_surface_targets": {
        "certified_active_personas": activated_kpis["certified_active_personas"],
        "deferred_external_personas": activated_kpis["deferred_external_personas"],
        "total_allowed_persona_actions": activated_kpis["total_allowed_persona_actions"],
    },
    "packaging_targets": {
        "active_persona_count": packaging_kpis["active_persona_count"],
        "planned_persona_count": packaging_kpis["planned_persona_count"],
        "total_persona_count": packaging_kpis["total_persona_count"],
        "total_routed_actions": packaging_kpis["total_routed_actions"],
    },
    "residual_gaps": residual_gaps,
}

hash_payload = json.dumps(verification, sort_keys=True, separators=(",", ":")).encode("utf-8")
verification["deterministic_hash"] = hashlib.sha256(hash_payload).hexdigest()

paths["verification_output"].parent.mkdir(parents=True, exist_ok=True)
paths["verification_output"].write_text(json.dumps(verification, indent=2, sort_keys=True) + "\n", encoding="utf-8")
verification


The correction must fail closed unless all three certification layers reconcile to the already-truthful final registry and packaging surfaces. Department pack status labels are intentionally preserved from the current pack contracts because those files are out of scope here.


In [ ]:
assert verification["predecessor_scope_gap_proof"] is True
assert verification["registry_summary"] == {
    "total_personas": 59,
    "total_departments": 10,
    "active_personas": 54,
    "planned_personas": 5,
    "registry_backed_personas": 54,
    "persona_contract_only_personas": 5,
}
assert verification["final_persona_certification_targets"] == {
    "total_personas": 59,
    "active_personas": 54,
    "planned_personas": 5,
    "registry_backed_personas": 54,
    "contract_only_personas": 5,
    "risk_tier_critical": 5,
    "risk_tier_high": 8,
    "risk_tier_medium": 14,
    "risk_tier_low": 32,
}
assert verification["final_department_certification_targets"]["total_departments"] == 10
assert verification["final_department_certification_targets"]["active_department_personas"] == 54
assert verification["final_department_certification_targets"]["planned_department_personas"] == 5
assert verification["final_department_certification_targets"]["registry_backed_department_personas"] == 54
assert verification["final_department_certification_targets"]["contract_only_department_personas"] == 5
assert verification["final_department_certification_targets"]["department_statuses"]["marketing"] == {
    "persona_count": 8,
    "active_persona_count": 3,
    "planned_persona_count": 5,
    "registry_backed_persona_count": 3,
    "contract_only_persona_count": 5,
    "workflow_family_count": 4,
    "workload_family_count": 7,
    "department_status": "partial-activation",
    "certification_verdict": "certified",
}
assert verification["final_department_certification_targets"]["department_statuses"]["operations"]["department_status"] == "partial-activation"
assert verification["final_department_certification_targets"]["department_statuses"]["design"]["department_status"] == "registry-backed"
assert verification["final_release_gate_targets"] == {
    "workload_domains_certified": 13,
    "workload_domains_not_yet": 2,
    "personas_certified": 59,
    "active_personas_certified": 54,
    "planned_personas_certified": 5,
    "departments_certified": 10,
    "total_routed_actions": 430,
}
assert verification["activated_surface_targets"] == {
    "certified_active_personas": 54,
    "deferred_external_personas": 5,
    "total_allowed_persona_actions": 430,
}
assert verification["packaging_targets"] == {
    "active_persona_count": 54,
    "planned_persona_count": 5,
    "total_persona_count": 59,
    "total_routed_actions": 430,
}
assert len(verification["residual_gaps"]) == 4
print("post-H5 unified certification parity targets proven")
